# Solutions — Nested Data: Orders, Items, Customers

Exercise notebook: `02_collections/05_nested_data.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
orders = [
    {"id": 101, "customer": "Pavan", "items": [{"name": "Laptop", "price": 70000}, {"name": "Mouse", "price": 1000}]},
    {"id": 102, "customer": "Diya", "items": [{"name": "Desk", "price": 12000, "qty": 1}, {"name": "Lamp", "price": 1500, "qty": 2}]},
    {"id": 103, "customer": "Pavan", "items": [{"name": "Coffee Beans", "price": 650, "qty": 4}]},
    {"id": 104, "customer": "Erik", "items": []},
]

## Exercise 1 — 🟢 Easy · Order totals

**Working solution**

In [ ]:
def order_totals(orders):
    return {o["id"]: sum(i["price"] * i.get("qty", 1) for i in o["items"]) for o in orders}

**Explanation**

A dict comprehension containing a generator expression: reads like the spec. `sum()` of an empty generator is `0`.

**Complexity:** O(total items).

**Common mistakes**

- `i['qty']` → `KeyError` for items without qty.

## Exercise 2 — 🟡 Medium · Customer lifetime spend ranking

**Working solution**

In [ ]:
from collections import defaultdict

def customer_ranking(orders):
    spend = defaultdict(float)
    count = defaultdict(int)
    for o in orders:
        spend[o["customer"]] += sum(i["price"] * i.get("qty", 1) for i in o["items"])
        count[o["customer"]] += 1
    return sorted(((c, spend[c], count[c]) for c in spend), key=lambda t: (-t[1], t[0]))

**Explanation**

Two accumulators keyed by customer, then one sort with a mixed-direction tuple key.

**Complexity:** O(items + c log c).

**Common mistakes**

- Sorting by `reverse=True` on the whole tuple — that also reverses the name tiebreak.

## Exercise 3 — 🟡 Medium · Explode nested orders into rows

**Working solution**

In [ ]:
def explode_items(orders):
    rows = []
    for o in orders:
        base = {"order_id": o["id"], "customer": o["customer"]}
        if not o["items"]:
            rows.append({**base, "line_no": 0, "item": None, "price": None, "qty": None, "line_total": None})
            continue
        for line_no, item in enumerate(o["items"], start=1):
            qty = item.get("qty", 1)
            rows.append({**base, "line_no": line_no, "item": item["name"], "price": item["price"],
                         "qty": qty, "line_total": item["price"] * qty})
    return rows

**Explanation**

`{**base, ...}` builds a new dict from shared parent fields plus item fields — no mutation, no copying bugs.

**Complexity:** O(total items).

**Common mistakes**

- Reusing one `row` dict and mutating it in the loop — every appended row becomes the last one.

## Exercise 4 — 🟠 Hard · Revenue by country and category from real JSON

**Working solution**

In [ ]:
from collections import defaultdict

def category_report(orders):
    report = defaultdict(lambda: defaultdict(float))
    for order in orders:
        if (order.get("payment") or {}).get("status") != "captured":
            continue
        country = order["customer"]["country"]
        for item in order["items"]:
            report[country][item.get("category") or "UNKNOWN"] += item["price"] * item["qty"]
    return {country: {cat: round(v, 2) for cat, v in sorted(cats.items())} for country, cats in sorted(report.items())}

**Explanation**

A nested `defaultdict(lambda: defaultdict(float))` builds a 2-level group-by without existence checks.
`(order.get("payment") or {}).get("status")` is a safe navigation idiom — Python's version of `Optional.map(...)`.

**Complexity:** O(total items).

**Common mistakes**

- Returning the `defaultdict` itself — printing/comparing works, but callers can accidentally create keys on read.

## Challenge — 🔴 Very Hard · Safe path access and field projection

**Working solution**

In [ ]:
_MISSING = object()

def get_path(obj, path, default=None):
    current = obj
    for part in path.split("."):
        if isinstance(current, dict):
            current = current.get(part, _MISSING)
        elif isinstance(current, list):
            try:
                current = current[int(part)]
            except (ValueError, IndexError):
                current = _MISSING
        else:
            current = _MISSING
        if current is _MISSING:
            return default
    return current


def project(records, mapping):
    return [{out: get_path(r, path) for out, path in mapping.items()} for r in records]

**Explanation**

A private sentinel object (`_MISSING`) distinguishes "key absent" from "value is None/0/''". This is the Pythonic answer to
chains of `Optional.ofNullable(x).map(...).map(...)`. Note `None` in the middle of a path falls into the `else` branch.

**Complexity:** O(depth) per lookup.

**Common mistakes**

- Using `None` as the 'missing' marker — then a real `None` value is indistinguishable from absence.
- `try: obj['a']['b'][0] except Exception:` — swallows real bugs.